# Project 02 — Apriori-TID Mining (Local)
Chạy từ checkout local với Julia và Java 25+ đã cài đặt. Không clone repository hoặc tự tải runtime.


## 1. Cài đặt môi trường
Chuẩn bị Julia, Java và các thư viện cần thiết.

In [ ]:
import os
import shutil
import shlex
import re
import subprocess
import tempfile
from pathlib import Path
def download_runtime_archive(url, destination):
    subprocess.run(["curl", "-fLsS", "--retry", "2", "--max-time", "240",
                    url, "-o", str(destination)], check=True)
from IPython.display import Image, SVG, Markdown, display

repository = next((directory for directory in [Path.cwd(), *Path.cwd().parents]
                   if (directory / "scripts/visual_pipeline.jl").is_file()), None)
if repository is None:
    raise RuntimeError("Open this notebook from a local checkout of apriori-tid-mining.")

for required_file in ["demo.jl", "scripts/fetch_datasets.jl", "scripts/plot_results.jl",
                      "scripts/visual_pipeline.jl", "scripts/run_all_tests.jl"]:
    if not (repository / required_file).is_file():
        raise RuntimeError("The repository is missing the integrated files. Upload the ZIP or update the GitHub branch.")

julia = shutil.which("julia")
if julia is None:
    raise RuntimeError("Install Julia 1.9+ and add it to PATH before running this notebook.")

# Cài Java 25 cho SPMF nếu cần.
def java_is_compatible():
    if shutil.which("java") is None:
        return False
    version = subprocess.run(["java", "-version"], capture_output=True, text=True, check=True)
    major = re.search(r'version "(\d+)', version.stderr + version.stdout)
    return major is not None and int(major.group(1)) >= 25

if not java_is_compatible():
    raise RuntimeError("Install Java 25+ and add it to PATH before running this notebook.")
subprocess.run(["java", "-version"], check=True)

workspace = Path(tempfile.mkdtemp(prefix="apriori-demo-"))
raw_data = repository / "data" / "raw"

def run_julia(environment, script, *arguments, extra_environment=None, threads=None):
    command = [julia, "--startup-file=no"]
    if threads is not None:
        command.append(f"--threads={threads}")
    command.extend([f"--project={repository / environment}", str(repository / script)])
    for argument in arguments:
        command.append(str(argument))
    print(shlex.join(command), flush=True)
    subprocess.run(command, cwd=repository, check=True, env=extra_environment)

def instantiate(environment):
    subprocess.run([julia, "--startup-file=no", f"--project={repository / environment}",
                    "-e", "using Pkg; Pkg.instantiate()"], cwd=repository, check=True,
                   env=os.environ | {"JULIA_NUM_PRECOMPILE_TASKS": "2"})

print(subprocess.check_output([julia, "--version"], text=True).strip())
print("Repository:", repository)
instantiate(".")
instantiate("evaluation_environment")
instantiate("visual_environment")


## 2. Kiểm tra Basic và Optimized
Chạy hai phiên bản với minsup 60% và so sánh kết quả.

In [ ]:
for implementation in ("basic", "optimized"):
    run_julia(".", "demo.jl", "--minsup", "0.6", repository / "data/toy/sample.txt",
              "--output", workspace / f"{implementation}.txt", "--implementation", implementation)

basic_result = (workspace / "basic.txt").read_text()
assert basic_result == (workspace / "optimized.txt").read_text()
print(basic_result)


## 3. Kiểm thử
Chạy bộ test thuật toán, SPMF và xử lý ảnh.


In [ ]:
run_julia(".", "test/runtests.jl")


## 4. Tải dữ liệu benchmark
Tải và kiểm tra bốn bộ dữ liệu FIMI.


In [ ]:
run_julia("evaluation_environment", "scripts/fetch_datasets.jl", "benchmark", "--root", raw_data)
benchmark_input_directory = raw_data / "benchmark"


### 4.1. So sánh với SPMF
So sánh itemset và support ở một mức minsup cho mỗi bộ dữ liệu.


In [ ]:
benchmark_quick_cases = [
    ("chess", "0.96"),
    ("mushroom", "0.75"),
    ("retail", "0.05"),
    ("T10I4D100K", "0.045"),
]
spmf_jar = repository / "tools/spmf.jar"
for dataset_name, minimum_support in benchmark_quick_cases:
    print(f"\n{dataset_name}: minsup = {minimum_support}", flush=True)
    run_julia(".", "scripts/compare_spmf.jl", spmf_jar,
              benchmark_input_directory / f"{dataset_name}.txt", minimum_support, threads=1)


### 4.2. Chạy benchmark
Đo thời gian, bộ nhớ và khả năng mở rộng. Đặt `run_full_benchmark = False` để bỏ qua bước này.

In [ ]:
run_full_benchmark = True  #@param {type:"boolean"}
if run_full_benchmark:
    benchmark_directory = workspace / "benchmark_new"
    run_julia(".", "scripts/run_benchmark.jl", benchmark_input_directory,
              benchmark_directory, spmf_jar, repository / "configs/benchmark.toml", threads=1)
    run_julia("evaluation_environment", "scripts/plot_results.jl", "benchmark", benchmark_directory)
    print("New benchmark results:", benchmark_directory)
    print((benchmark_directory / "environment.txt").read_text())
    for filename in ("runtime_ms.png", "itemsets.png", "peak_ram.png", "scaling_and_width.png"):
        display(Image(filename=str(benchmark_directory / filename)))
else:
    print("Skipped full benchmark, set run_full_benchmark = True to run it.")


## 5. Ứng dụng 1 — Market Basket Analysis
Tạo luật kết hợp từ Groceries với minsup 1% và minconf 40%.

In [ ]:
groceries_output = workspace / "groceries"
run_julia("evaluation_environment", "scripts/fetch_datasets.jl", "groceries", "--root", raw_data)
run_julia(".", "scripts/run_market_basket.jl", raw_data / "groceries/Groceries.data",
          raw_data / "groceries/Groceries.labels", groceries_output, "0.01", "0.4", "optimized")
run_julia("evaluation_environment", "scripts/plot_results.jl", "groceries", groceries_output)
print((groceries_output / "top10_rules.csv").read_text())
for filename in ("basket_lengths.png", "top10_rules.png"):
    display(Image(filename=str(groceries_output / filename)))

## 6. Ứng dụng 2 — Visual Pattern Mining
Chuyển ảnh Flowers-17 thành visual words và khai thác tập phổ biến.
Mặc định chạy toàn bộ ảnh, đổi `image_limit` để chạy thử ít ảnh hơn.

In [ ]:
image_limit = 0
visual_words = 8
maximum_descriptors = 100
minimum_support_values = "0.7,0.8,0.9"
visual_output = workspace / "flowers_k8"

run_julia("visual_environment", "scripts/visual_pipeline.jl", "--output", visual_output,
          "--limit", image_limit, "--words", visual_words, "--maximum-descriptors", maximum_descriptors,
          "--seed", 42, "--minsup", minimum_support_values)
print((visual_output / "dataset_summary.toml").read_text())
print((visual_output / "runs.csv").read_text())
display(Markdown((visual_output / "analysis.md").read_text()))
for figure_path in sorted((visual_output / "figures").glob("*.svg")):
    display(SVG(filename=str(figure_path)))

## 7. Thử cấu hình ảnh khác
Chạy lại với 16 cụm và tối đa 40 descriptor mỗi ảnh.

In [ ]:
alternative_output = workspace / "flowers_k16"
run_julia("visual_environment", "scripts/visual_pipeline.jl", "--output", alternative_output,
          "--limit", image_limit, "--words", 16, "--maximum-descriptors", 40,
          "--seed", 42, "--minsup", minimum_support_values)
print((alternative_output / "dataset_summary.toml").read_text())
print((alternative_output / "runs.csv").read_text())
display(Markdown((alternative_output / "analysis.md").read_text()))
for figure_path in sorted((alternative_output / "figures").glob("*.svg")):
    display(SVG(filename=str(figure_path)))

## 8. Thư mục kết quả
Đường dẫn đến các kết quả vừa chạy.

In [ ]:
print("Output directory:", workspace)